In [1]:
import os
import pandas as pd
import scipy
import cobra
from cobra.io import load_matlab_model
from cobra.io import read_sbml_model
from cobra.flux_analysis import gapfill
from cobra.io import write_sbml_model
from cobra.io import save_matlab_model
from cobra.flux_analysis import fastcc
from cobra.flux_analysis import flux_variability_analysis

# Investigating the unconnected metabolites identified from memote report in consensus brain model

In [2]:
# read the consensus model and the mouse GEM model
model = load_matlab_model('/Users/egabal/Library/CloudStorage/Box-Box/PFOS_Project/drafts_statistics_AND_consensus_draft/models/iMiceBrain_test1.mat')
universal = load_matlab_model('/Users/egabal/Library/CloudStorage/Box-Box/PFOS_Project/mouse_model_iMM1865/iMM1865_updated.mat')

Set parameter Username
Set parameter LicenseID to value 2685327
Academic license - for non-commercial use only - expires 2026-07-07


No defined compartments in model model. Compartments will be deduced heuristically using regular expressions.
Using regular expression found the following compartments:c, e, g, i, l, m, n, r, x


In [30]:
# investigate the low connectivity percentage of metabolites from first report from memote
disconnected_mets = [met for met in model.metabolites if not met.reactions]

print(f"Found {len(disconnected_mets)} disconnected metabolites.")

# print these mets
for met in disconnected_mets:
    print(f"{met.id}: {met.name}")

Found 0 disconnected metabolites.


In [28]:
from cobra.flux_analysis import pfba
sol = pfba(model)
print(sol.fluxes['ATPS4mi'])
print(sol.fluxes['GLUt6'])
print(sol.fluxes['BIOMASS_reaction'])

100.0
100.0
44.027758896057264


In [5]:
print("Reactions:", len(model.reactions))
print("Genes:", len(model.genes))
print("Metabolites:", len(model.metabolites))

Reactions: 6040
Genes: 1856
Metabolites: 3111


# adding other annotation to genes 

In [7]:
for i in range (len(model.reactions)):
    if model.reactions[i].id.startswith('EX_'):
        model.reactions[i].bounds = (-100, 100)

In [8]:
# read annotation file and pass the values as quoted string
annotations_genes = pd.read_csv('/Users/egabal/Desktop/iMiceBrain_genes_annotation_polished.csv')
print(annotations_genes.iloc[0, 0])

227620


In [9]:
# add these information to the model as structures 
# change the format for ccds and refseq to match the format of memote
for gene, row in zip(model.genes, annotations_genes.itertuples(index=False)):
    # Make sure annotation is a dict
    if gene.annotation is None:
        gene.annotation = {}

    # Update or add fields without overwriting unrelated keys
    gene.annotation.update({
        'ncbigene': str(row.entrezgene_id) if pd.notna(row.entrezgene_id) else '',
        'refseq': str(row.refseq_mrna) if pd.notna(row.refseq_mrna) else '',
        'uniprot': str(row.uniprot_gn_id) if pd.notna(row.uniprot_gn_id) else '',
        'kegg.genes': str(row.kegg_gene_ids) if pd.notna(row.kegg_gene_ids) else '',
        'ensembl': str(row.ensembl_gene_id) if pd.notna(row.ensembl_gene_id) else '',
        'ccds': str(row.ccds) if pd.notna(row.ccds) else ''
        })

In [6]:
# update model compartments 
model.compartments = {
    'r': 'Endoplasmic Reticulum',
    'c': 'Cytoplasm',
    'l': 'Lysosome',
    'm': 'Mitochondrion',
    'e': 'Extracellular',
    'g': 'Golgi apparatus',
    'x': 'Peroxisome',
    'n': 'Nucleus',
    'i': 'inner mitochondrial membrane'
}
model.name = "iMiceBrain_GEM Baloni Lab 2025"

# update the metabolites annotations as well (will export the model with new annotations after covering all missing IDs from other databases)

In [6]:
# adding the pubchem ids for metabolites
mets = pd.read_csv('/Users/egabal/Desktop/iMiceBrain_metabolites_annotation.csv')

# update the mets annotations by adding the PUBchem ids
for idx, row in mets.iterrows():
    met_id = row['Met_ID']
    try:
        met = model.metabolites.get_by_id(met_id)
        annotations = {}

        # PubChem
        if pd.notna(row['PubChemID']):
            annotations['pubchem.compound'] = str(int(row['PubChemID']))

        # HMDB
        if pd.notna(row['HMDB_ID']):
            annotations['hmdb'] = str(row['HMDB_ID'])

        # InChI and InChIKey
        if pd.notna(row['InChI_ID']):
            annotations['inchi'] = row['InChI_ID']
        if pd.notna(row['InChIKey']):
            annotations['inchikey'] = row['InChIKey']

        # KEGG
        if pd.notna(row['KEGG_ID']):
            annotations['kegg.compound'] = row['KEGG_ID']

        # BioCyc
        if pd.notna(row['BioCyc_ID']):
            annotations['biocyc'] = row['BioCyc_ID']

        # Reactome
        if pd.notna(row['Reactome_IDs']):
            annotations['reactome'] = row['Reactome_IDs']

        # Assign all annotations at once
        met.annotation.update(annotations)

    except KeyError:
        print(f"Metabolite {met_id} not found in model.")

# Add the reactions annotation to the model


In [3]:
# adding rxns annotations
rxns = pd.read_csv("/Users/egabal/Desktop/iMiceBrain_reactions_with_SBO.csv")

for idx, row in rxns.iterrows():
    rxn_id = row['bigg']  

    try:
        rxn = model.reactions.get_by_id(rxn_id)
        annotations = {}

        if pd.notna(row['metanetx_reaction']):
            annotations['metanetx.reaction'] = row['metanetx_reaction']

        if pd.notna(row['seed_reaction']):
            annotations['seed.reaction'] = row['seed_reaction']

        if pd.notna(row['ec_numbers']):
            annotations['ec-code'] = row['ec_numbers']

        if pd.notna(row['KEGG']):
            annotations['kegg.reaction'] = row['KEGG']

        if pd.notna(row['Assigned_SBO']):
            annotations['sbo'] = row['Assigned_SBO']

        rxn.annotation.update(annotations)

    except KeyError:
        print(f"Reaction {rxn_id} not found in model.")

In [7]:
# export the models after refining and then test through memote
from cobra.io import write_sbml_model
from cobra.io import save_matlab_model

save_matlab_model(model, "/Users/eso1993/Desktop/iMiceBrain_APOE_GEM_2.mat")
write_sbml_model(model, "/Users/eso1993/Desktop/iMiceBrain_APOE_GEM_2.xml")

Error encountered trying to <Setting notes on sbase: <Reaction R_IDL_HSSYN "Formation of IDL">>.
LibSBML error code -3: The requested action could not be performed. This can occur in a variety of contexts, such as passing a null object as a parameter in a situation where it does not make sense to permit a null object.
Error encountered trying to <Setting notes on sbase: <Reaction R_LDL_HSSYN "Formation of LDL">>.
LibSBML error code -3: The requested action could not be performed. This can occur in a variety of contexts, such as passing a null object as a parameter in a situation where it does not make sense to permit a null object.
Error encountered trying to <Setting notes on sbase: <Reaction R_HDL_HSSYN "Formation of HDL">>.
LibSBML error code -3: The requested action could not be performed. This can occur in a variety of contexts, such as passing a null object as a parameter in a situation where it does not make sense to permit a null object.
Error encountered trying to <Setting not

# Working on some SBO annotations in the model according to the memote report and modifiying some reactions

In [2]:
model = load_matlab_model('/Users/egabal/Library/CloudStorage/Box-Box/PFOS_Project/drafts_statistics_AND_consensus_draft/models/final_models/iMiceBrain.mat')

Set parameter Username
Set parameter LicenseID to value 2685327
Academic license - for non-commercial use only - expires 2026-07-07


In [8]:
# ✅ Assign SBO:0000632 for sink reactions (preserve existing annotations)
for rxn in model.reactions:
    if rxn.id.startswith("SK_"):
        rxn.annotation.update({"sbo": "SBO:0000632"})

# ✅ Annotate genes with general SBO term for genes: SBO:0000243
# (preserve existing annotations, only update if different)
updated = 0
for g in model.genes:
    if g.annotation.get("sbo") != "SBO:0000243":
        g.annotation.update({"sbo": "SBO:0000243"})
        updated += 1

# ✅ Assign SBO:0000629 to biomass reactions (preserve existing annotations)
for rxn in model.reactions:
    if rxn.id.startswith("BIOMASS_"):
        rxn.annotation.update({"sbo": "SBO:0000629"})


In [4]:
# add a demand reaction for DM_nadph_c and DM_atp[c]

from cobra import Reaction

met_id = "atp[c]"

# Get the metabolite
met = model.metabolites.get_by_id(met_id)

# Create the demand reaction
rxn_id = f"DM_{met_id.replace('[','_').replace(']','')}"
demand_rxn = Reaction(rxn_id)
demand_rxn.name = f"Demand for {met.name}"
demand_rxn.subsystem = "Exchange/demand reaction"
demand_rxn.lower_bound = 0.0  # irreversible: only consumption
demand_rxn.upper_bound = 1000.0
demand_rxn.add_metabolites({met: -1.0})  # consumes 1 unit of the metabolite

# Annotate as demand reaction
demand_rxn.annotation['sbo'] = 'SBO:0000628'  # SBO term for "demand reaction"

# Add to the model
model.add_reactions([demand_rxn])

print("Added:", demand_rxn.id, "with SBO:", demand_rxn.annotation['sbo'], "for", met.id)

Ignoring reaction 'DM_atp_c' since it already exists.


Added: DM_atp_c with SBO: SBO:0000628 for atp[c]


In [9]:
# changing the model id and dsecritpion
model.id = "iMiceBrain"  # Unique identifier for the model
model.name = "mouse brain-specific GEM" 
model.version = "v2025"
model.organism = "Mus musculus"
model.tissue = "Brain"
model.creator = "Baloni Lab"

In [10]:
model.annotation = {
    "biosource": "brain",
    "organism": "Mus musculus",
    "creator": "Baloni Lab",
    "version": "2025",
    "description": "iMiceBrain-Baloni-Lab-2025"
}

In [11]:
save_matlab_model(model, "/Users/egabal/Desktop/iMiceBrain.mat")
write_sbml_model(model, "/Users/egabal/Desktop/iMiceBrain.xml")
